# Module 4: Supply Chains - Sourcing and Material Reality
## REE 4301 / IE 5300 - Energy Systems Modeling

Two halves, and you need both.

**The first is a model.** Where should a battery manufacturer buy its cobalt, given what each mine can produce, what each refinery can process, and what every route costs? That is a decision, and a linear program makes it.

**The second is accounting.** Given a build plan - so many GW of wind, solar and storage - how much steel, copper, lithium and nickel does it actually require? Nothing is optimised; it is multiplication. But it is the question most of you will be handed first, because procurement comes before strategy in almost every career.

> **You have already written the first half.** The sourcing problem is the Module 3 transport LP with different labels. That is not laziness on my part - noticing it is the skill.


In [ ]:
!pip install -q pypsa highspy gurobipy


In [ ]:
import numpy as np
import pandas as pd
import gurobipy as gp
from gurobipy import GRB
import pypsa
import warnings
warnings.filterwarnings('ignore')

SOLVER = 'gurobi'
# SOLVER = 'highs'     # <-- uncomment: open source, no licence, no size cap

WLS = {}   # {'WLSACCESSID': '...', 'WLSSECRET': '...', 'LICENSEID': 000000}
ENV = gp.Env(params=WLS) if (SOLVER == 'gurobi' and WLS) else None

print(f'solver: {SOLVER}'
      + ('  (academic WLS licence)' if ENV else '  (default licence)'))


---
# Part A - The sourcing problem, on paper

A battery manufacturer needs **120 kt of refined cobalt a year** across two cell plants. Ore comes from three mines; it has to be refined at one of two processors before a cell plant can use it.

| mine | capacity kt/yr | | processor | capacity kt/yr | | plant | needs |
|---|---|---|---|---|---|---|---|
| DRC | 120 | | China | 120 | | Cell Plant A | 70 |
| Australia | 60 | | Domestic | 120 | | Cell Plant B | 50 |
| Domestic | 40 | | | | | | |

Costs are dollars per kt, and they cover shipping plus tolling:

| mine to processor | China | Domestic |
|---|---|---|
| DRC | 2.0 | 6.0 |
| Australia | 4.0 | 5.5 |
| Domestic | 8.0 | 3.0 |

| processor to plant | Plant A | Plant B |
|---|---|---|
| China | 3.0 | 3.0 |
| Domestic | 5.0 | 5.0 |

**Predict before you read on.** Where does the cheapest answer buy from? Write down a rough split across the three mines.


### The five parts

**Sets.** *i* mines, *p* processors, *j* plants.

**Parameters.** *M<sub>i</sub>* mine capacity, *P<sub>p</sub>* processor capacity, *D<sub>j</sub>* plant demand, *a<sub>ip</sub>* and *b<sub>pj</sub>* the costs on each leg.

**Decision variables.** *x<sub>ip</sub>* ore from mine *i* to processor *p*; *y<sub>pj</sub>* refined metal from processor *p* to plant *j*. Both non-negative.

**Objective.** Minimise Σ *a<sub>ip</sub> x<sub>ip</sub>* + Σ *b<sub>pj</sub> y<sub>pj</sub>*.

**Constraints.**
- mine capacity: Σ<sub>p</sub> *x<sub>ip</sub>* ≤ *M<sub>i</sub>*
- processor capacity: Σ<sub>i</sub> *x<sub>ip</sub>* ≤ *P<sub>p</sub>*
- **conservation at the processor**: Σ<sub>i</sub> *x<sub>ip</sub>* = Σ<sub>j</sub> *y<sub>pj</sub>* — what goes in comes out
- demand: Σ<sub>p</sub> *y<sub>pj</sub>* = *D<sub>j</sub>*

That conservation constraint is the only structural difference from the transport problem, and it is the same `Bus-nodal_balance` you have met twice already. A refinery is a bus.


---
# Part B - In gurobipy, five parts numbered

Same layout as the transport and power-flow companions.


In [ ]:
# ==========================================
# 1. Sets and Parameters (Data)
# ==========================================
mines = {'DRC': 120.0, 'Australia': 60.0, 'Domestic': 40.0}     # M_i, kt/yr
procs = {'China': 120.0, 'Domestic': 120.0}                     # P_p, kt/yr
plants = {'Cell Plant A': 70.0, 'Cell Plant B': 50.0}           # D_j, kt/yr

# a_ip : $/kt, mine -> processor (shipping + tolling)
ship = {('DRC', 'China'): 2.0,       ('DRC', 'Domestic'): 6.0,
        ('Australia', 'China'): 4.0, ('Australia', 'Domestic'): 5.5,
        ('Domestic', 'China'): 8.0,  ('Domestic', 'Domestic'): 3.0}

# b_pj : $/kt, processor -> plant
deliver = {('China', 'Cell Plant A'): 3.0, ('China', 'Cell Plant B'): 3.0,
           ('Domestic', 'Cell Plant A'): 5.0,
           ('Domestic', 'Cell Plant B'): 5.0}

# ==========================================
# 2. Model Initialization
# ==========================================
m = gp.Model('sourcing', env=ENV) if ENV else gp.Model('sourcing')
m.Params.OutputFlag = 0


In [ ]:
# ==========================================
# 3. Decision Variables
# ==========================================
x = m.addVars(ship.keys(), lb=0, name='x')       # ore, mine -> processor
y = m.addVars(deliver.keys(), lb=0, name='y')    # metal, processor -> plant

# ==========================================
# 4. Objective Function
# ==========================================
m.setObjective(x.prod(ship) + y.prod(deliver), GRB.MINIMIZE)


In [ ]:
# ==========================================
# 5. Constraints
# ==========================================
for i, cap in mines.items():
    m.addConstr(x.sum(i, '*') <= cap, f'mine_{i}')

for p, cap in procs.items():
    m.addConstr(x.sum('*', p) <= cap, f'proc_cap_{p}')
    # conservation: a refinery cannot ship what it did not receive.
    # This is Bus-nodal_balance wearing a different hat.
    m.addConstr(x.sum('*', p) == y.sum(p, '*'), f'balance_{p}')

for j, need in plants.items():
    m.addConstr(y.sum('*', j) == need, f'demand_{j}')


In [ ]:
# ==========================================
# 6. Optimize and Output
# ==========================================
m.optimize()
assert m.Status == GRB.OPTIMAL
base_cost = m.ObjVal

by_mine = {i: sum(x[i, p].X for p in procs) for i in mines}
by_proc = {p: sum(y[p, j].X for j in plants) for p in procs}
total = sum(plants.values())

print(f'total cost ${base_cost:,.2f}')
print()
print('bought from')
for i, v in by_mine.items():
    print(f'  {i:12s} {v:6.1f} kt   {v / total:5.0%}')
print('refined at')
for p, v in by_proc.items():
    print(f'  {p:12s} {v:6.1f} kt   {v / total:5.0%}')


---
# Part C - Read that answer again

**Everything comes from one mine, through one processor.** Not most of it - all of it.

Before reading on: is that a bug, an artefact of numbers too small to be interesting, or the model doing exactly what you asked it to?

The real cobalt chain looks much like your answer - a majority of mined supply from one country, a large majority of refining in another. Nobody chose that as a strategy; a sequence of individually rational least-cost decisions arrived at it. What does your objective function have in common with that sequence?

### You have written this model twice before

| transport companion | this notebook |
|---|---|
| supply node, `Generator.p_nom` | mine, capacity *M<sub>i</sub>* |
| demand node, `Load.p_set` | cell plant, demand *D<sub>j</sub>* |
| pipeline tariff *c<sub>ij</sub>* | shipping + tolling *a<sub>ip</sub>* |
| pipeline capacity | processor capacity *P<sub>p</sub>* |
| nodal balance | conservation at the refinery |

Same five parts, same solver, same structure - crude oil in one, cobalt in the other. **If you can see that, you can model a supply chain you have never been taught.** That transferability is the point of the whole course, and this is the cheapest place to notice it.


---
# Part D - What does it cost not to be concentrated?

A procurement officer cannot accept a single point of failure, whatever the spreadsheet says. One expropriation, one export ban, one shipping route closed, and production stops.

So add a constraint: **no single mine may supply more than a given share of total demand.** One line of algebra.

&nbsp;&nbsp;&nbsp;&nbsp;Σ<sub>p</sub> *x<sub>ip</sub>* ≤ *s* · Σ<sub>j</sub> *D<sub>j</sub>*&nbsp;&nbsp; for every mine *i*

**Predict first.** At a 50% cap, roughly how much more do you expect the cobalt to cost - a few per cent, or tens of per cent?


In [ ]:
def solve_sourcing(share_cap=None):
    """Re-solve with an optional single-source cap. Returns (cost, mix)."""
    k = gp.Model(env=ENV) if ENV else gp.Model()
    k.Params.OutputFlag = 0
    xx = k.addVars(ship.keys(), lb=0)
    yy = k.addVars(deliver.keys(), lb=0)
    k.setObjective(xx.prod(ship) + yy.prod(deliver), GRB.MINIMIZE)
    for i, cap in mines.items():
        k.addConstr(xx.sum(i, '*') <= cap)
    for p, cap in procs.items():
        k.addConstr(xx.sum('*', p) <= cap)
        k.addConstr(xx.sum('*', p) == yy.sum(p, '*'))
    for j, need in plants.items():
        k.addConstr(yy.sum('*', j) == need)
    if share_cap is not None:
        for i in mines:
            k.addConstr(xx.sum(i, '*') <= share_cap * sum(plants.values()))
    k.optimize()
    if k.Status != GRB.OPTIMAL:
        return None, None
    return k.ObjVal, {i: sum(xx[i, p].X for p in procs) for i in mines}


rows = []
for cap in [None, 0.75, 0.60, 0.50, 0.40, 0.34]:
    cost, mix = solve_sourcing(cap)
    label = 'no cap' if cap is None else f'{cap:.0%}'
    if cost is None:
        rows.append({'cap': label, 'cost': None, 'premium': 'INFEASIBLE'})
        continue
    rows.append({'cap': label, 'cost': round(cost, 2),
                 'premium': f'{cost / base_cost - 1:.1%}',
                 **{i: round(v, 1) for i, v in mix.items()}})

print(pd.DataFrame(rows).to_string(index=False))


> **This is a price curve for resilience, and you can read it off the table.** Capping any one mine at 75% costs 10%. At 50% it costs 20%, and Australia enters the mix. At 40% it costs 28% and the domestic mine - the most expensive tonne in the problem - finally gets bought.
>
> Nothing here says which cap is right. That is a judgement about how much disruption risk is worth, and it belongs to a person. **What the model does is stop the argument being about whether diversification costs anything, and make it about how much.** That is usually the more useful conversation.
>
> **Exercise D.1.** Below about 34% the problem becomes infeasible. Say why in one sentence, without re-running it.
>
> **Exercise D.2.** Put the cap on the *processor* instead of the mine. Which constraint is more expensive to satisfy, and what does that tell you about where the real bottleneck in this industry sits?
>
> **Exercise D.3.** The domestic mine costs $8.0/kt to ship to China and $3.0 to the domestic processor. Find the shipping cost at which it enters the unconstrained solution on price alone, with no cap at all.


---
# Part E - The same problem in PyPSA

A supply chain maps onto PyPSA's components without strain, because they were built for exactly this shape:

- a **Bus** is a place where a balance holds - a mine, a refinery, a plant
- a **Generator** on a mine bus is that mine's production, capped at `p_nom`
- a **Link** is a route with a cost and a capacity
- a **Load** on a plant bus is what it must receive

The conservation constraint you wrote by hand at each refinery is what PyPSA gives you for free by making the refinery a bus.


In [ ]:
n = pypsa.Network()
n.set_snapshots([0])

# Bus.name maps to a set element: mines, processors and plants alike
for b in list(mines) + [f'{p} refinery' for p in procs] + list(plants):
    n.add('Bus', b)

# Generator.p_nom maps to the mine capacity M_i
for i, cap in mines.items():
    n.add('Generator', f'{i} mine', bus=i, p_nom=cap, marginal_cost=0.0)

# Load.p_set maps to the plant requirement D_j
for j, need in plants.items():
    n.add('Load', f'{j} demand', bus=j, p_set=need)

# Link.marginal_cost maps to the tariff; Link.p_nom to a route limit
for (i, p), c in ship.items():
    n.add('Link', f'{i}->{p}', bus0=i, bus1=f'{p} refinery',
          p_nom=1e4, efficiency=1.0, marginal_cost=c)
for (p, j), c in deliver.items():
    n.add('Link', f'{p}->{j}', bus0=f'{p} refinery', bus1=j,
          p_nom=1e4, efficiency=1.0, marginal_cost=c)

# processor capacity is a limit on the refinery's throughput, so it goes
# on the links INTO it - there is no component for 'a refinery' as such
for p, cap in procs.items():
    for i in mines:
        pass   # per-route limits would go here; the cap is enforced below

n.optimize(solver_name=SOLVER, env=ENV)
print(f'PyPSA cost ${n.objective:,.2f}   gurobipy cost ${base_cost:,.2f}')
assert abs(n.objective - base_cost) < 1e-6, 'the two models disagree'
print()
print(n.links_t.p0.iloc[0].round(1).to_string())


Same number, to the cent. As always, PyPSA did not invent new mathematics; it wrote your constraints from a description of the system.

> **Note what is missing.** The processor capacity is not enforced in the PyPSA version above, because a refinery here is a bus and a bus has no capacity. In this instance it happens not to bind, so the answers agree. **Making them disagree is Exercise E.1**: drop `China` capacity to 60 in the gurobipy model and re-solve, then work out where that limit has to go in PyPSA. (Hint: it is a property of the links, not of the bus.)


---
# Part F - The other half: what is it all made of?

Nothing is optimised from here on. This is accounting, and it is the question you are most likely to be handed in your first job: **a build plan exists; what does it require?**

A capacity plan in GW becomes a materials requirement by multiplying it by an intensity matrix - kilograms of each material per MW of each technology. The matrix below is simplified from IEA figures.

The plan used here is deliberately **decade-scale and national**, not one project. That is what makes the two halves of this notebook meet: a single wind farm's cobalt requirement is a rounding error, and only a programme generates a sourcing problem worth optimising.


In [ ]:
# kg of material per MW built.  Simplified from IEA material-intensity
# figures for illustration; cite the current edition in your own work.
intensity = pd.DataFrame({
    'steel':    {'Wind': 8000, 'Solar': 3000, 'Battery': 1000},
    'copper':   {'Wind': 1000, 'Solar': 3000, 'Battery': 5000},
    'aluminum': {'Wind':  500, 'Solar': 1000, 'Battery': 1000},
    'lithium':  {'Wind':    0, 'Solar':    0, 'Battery': 1500},
    'nickel':   {'Wind':    0, 'Solar':    0, 'Battery': 4000},
    'cobalt':   {'Wind':    0, 'Solar':    0, 'Battery':  700},
})
print(intensity.to_string())


In [ ]:
# A build plan. In a real study this comes OUT of a capacity-expansion
# model - SB6's p_nom_opt, for instance - rather than being typed here.
#
# These are decade-scale national numbers, not one project: the point of
# Part F is that a build TARGET implies a material requirement, and the
# requirement only becomes a sourcing problem at programme scale.
plan_mw = pd.Series({'Wind': 250_000, 'Solar': 400_000,
                     'Battery': 171_000})

# kg per MW  x  MW  ->  kg, then to kilotonnes
materials_kt = intensity.mul(plan_mw, axis=0).sum() / 1e6

out = pd.DataFrame({'kt required': materials_kt.round(1)})
out['share of build'] = (materials_kt / materials_kt.sum()).map('{:.1%}'.format)
print(f'build plan: {plan_mw.to_dict()}  MW')
print()
print(out.to_string())


### Now connect the two halves

The cobalt line in that table is the demand figure that Part A took as given. **A capacity plan is a supply-chain requirement**, and until you have multiplied it out, a build target is a sentence rather than a plan.


In [ ]:
cobalt_kt = float(materials_kt['cobalt'])
print(f'this build needs {cobalt_kt:,.1f} kt of cobalt')
print(f'Part A sourced       {sum(plants.values()):,.1f} kt')
print()
scale = cobalt_kt / sum(plants.values())
cost50, _ = solve_sourcing(0.50)
print(f'that is {scale:.2f}x the sourcing problem in Part A - which is to'
      f' say, essentially the same problem.')
print()
print(f'  sourced at least cost      ${base_cost * scale:>10,.0f}')
print(f'  with a 50% single-source cap ${cost50 * scale:>10,.0f}')
print(f'  the resilience premium     ${(cost50 - base_cost) * scale:>10,.0f}')


> **Exercise F.1.** Change the build plan to 30,000 MW of battery and nothing else. Which material becomes binding first against real world production, and how would you check that claim?
>
> **Exercise F.2.** The intensity matrix has one number per technology. Name two things that number hides - and say whether either would change the ranking of materials by requirement.
>
> **Exercise F.3 - the one worth writing up.** You now have both halves: a build plan implies a material requirement, and a material requirement implies a sourcing decision with a resilience premium. Take a 20 GW battery programme through both steps and state the annual cobalt cost under no cap and under a 50% cap. That number is a procurement recommendation.


---

*Before class: you imposed a limit on how much could come from any one source. At a single site that limit usually arrives as a contract rather than a choice. What would you pay to keep it?*



### Sources
- Material intensities simplified from IEA, *The Role of Critical Minerals in Clean Energy Transitions*. Cite the current edition in your own work; these are illustrative.
- Mine, refinery and cost figures in Parts A to E are invented to be hand-checkable. The **pattern** they produce - concentrated mining, more concentrated refining - is not invented.
- The transport companion this reuses: `M3_Transport_Companion.ipynb`.
